# CICIDS2017 PyTorch INSOMNIA-NC

Faithful PyTorch implementation of the INSOMNIA pipeline for CICIDS-2017: DNN detector, NearestCentroid label estimator, uncertainty sampling, pseudo-labeling, incremental fine-tuning, and count-based window evaluation.


## 2. Install/import dependencies

Kaggle notebooks include the core scientific Python stack. The install cell is intentionally minimal and can be extended if your runtime is missing a package.

In [ ]:
# Kaggle usually has these installed. Uncomment if your runtime is missing packages.
# !pip install -q pandas numpy scikit-learn torch

In [ ]:
import os
import json
import gc
import math
import pickle
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
DATASET_NAME = "CICIDS-2017"
FRAMEWORK = "PyTorch"
NOTEBOOK_FILENAME = "CICIDS2017_PyTorch_INSOMNIA.ipynb"
INSOMNIA_MODEL_NAME = "INSOMNIA-NC"
NO_UPDATE_MODEL_NAME = "No-Update DNN"
DNN_DETECTOR_NAME = "DNN detector"


## 3. Set reproducibility configuration

The training seeds and weight initialization tuples follow the reproducibility setup from “Randomness Unmasked: Towards Reproducible and Fair Evaluation of Shift-Aware Deep Learning NIDS”.

In [ ]:
TRAINING_SEEDS = [57, 305, 5, 9667, 405]
WEIGHT_INIT_SEEDS = {
    "W1": [1004, 77, 259, 35],
    "W2": [8, 358, 200, 35],
    "W3": [487, 22, 900, 7],
}

DEFAULT_SEED = TRAINING_SEEDS[0]

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_global_seed(DEFAULT_SEED)

## 4. Create data and Kaggle results folders\n\nGenerated CSV results are written to `/kaggle/working/results/`.\n

In [ ]:
DATA_DIR = Path("/kaggle/working/data")
RESULTS_DIR = Path("/kaggle/working/results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Final dataset path:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_PyTorch_INSOMNIA_results.csv"
AGGREGATED_RESULTS_CSV_PATH = RESULTS_DIR / "CICIDS2017_PyTorch_INSOMNIA_aggregated_results.csv"

print("Per-run results CSV:", RESULTS_CSV_PATH)
print("Aggregated results CSV:", AGGREGATED_RESULTS_CSV_PATH)


## 5. Dataset download with kagglehub

This notebook downloads the Kaggle dataset with `kagglehub`, copies the downloaded files into `/kaggle/working/data`, preserves subdirectories, and prints discovered CSV files.

In [ ]:
# Kaggle notebook equivalent: !pip install -q kagglehub
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

In [ ]:
import shutil
import kagglehub

path = kagglehub.dataset_download("bertvankeulen/cicids-2017")
print("Path to dataset files:", path)

source_path = Path(path)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if source_path.is_file():
    target_path = DATA_DIR / source_path.name
    if source_path.resolve() != target_path.resolve():
        shutil.copy2(source_path, target_path)
else:
    for source_item in source_path.rglob("*"):
        if source_item.is_file():
            relative_path = source_item.relative_to(source_path)
            target_path = DATA_DIR / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            if source_item.resolve() != target_path.resolve():
                shutil.copy2(source_item, target_path)

csv_files = sorted(DATA_DIR.rglob("*.csv"))
print("Final dataset path:", DATA_DIR)
print(f"CSV files discovered after copying: {len(csv_files)}")
for csv_file in csv_files:
    print(csv_file)

## 6. Load dataset from /kaggle/working/data

This loader recursively finds CSV files with `Path("/kaggle/working/data").rglob("*.csv")`, adds a `source_file` column before concatenation, prints the discovered files, reports dataset shape, and prints label distribution.

In [ ]:
LABEL_CANDIDATES = ["Label", "label", "Class", "class", "Attack", "attack"]
CICIDS_ORIGINAL_WEEKDAYS = ("monday", "tuesday", "wednesday", "thursday", "friday")


def detect_label_column(df: pd.DataFrame) -> str:
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate

    lower_map = {col.lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    object_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    object_columns = [col for col in object_columns if col != "source_file"]
    if object_columns:
        return object_columns[-1]

    raise ValueError(
        "Could not automatically detect a label column. Rename the target column to one of: "
        + ", ".join(LABEL_CANDIDATES)
    )


def is_original_cicids_file(csv_path: Path) -> bool:
    name = csv_path.name.lower()
    return "_plus" not in name and any(day in name for day in CICIDS_ORIGINAL_WEEKDAYS)


def load_csv_dataset(data_dir: Path) -> pd.DataFrame:
    csv_files = sorted(data_dir.rglob("*.csv"))
    csv_files = [csv_path for csv_path in csv_files if is_original_cicids_file(csv_path)]

    print("=== DATASET FILES USED ===")
    print(f"CSV files found after excluding *_plus.csv files: {len(csv_files)}")
    for csv_path in csv_files:
        print(" -", csv_path.relative_to(data_dir))

    if not csv_files:
        raise FileNotFoundError(
            "No original CICIDS-2017 CSV files found under /kaggle/working/data after excluding *_plus.csv files."
        )

    frames = []
    for csv_path in csv_files:
        print(f"Loading {csv_path}")
        frame = pd.read_csv(csv_path, low_memory=False)
        frame.columns = frame.columns.astype(str).str.strip()
        frame["source_file"] = str(csv_path.relative_to(data_dir))
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    df.columns = df.columns.astype(str).str.strip()

    print(f"Total final dataframe shape: {df.shape}")
    print("Column names:")
    print(list(df.columns))

    label_col = detect_label_column(df)
    print(f"Detected label column: {label_col}")
    print("Label distribution:")
    print(df[label_col].astype(str).str.strip().value_counts(dropna=False))
    return df


raw_df = load_csv_dataset(DATA_DIR)
raw_df.head()


## INSOMNIA Dataset Protocol and Binary Preprocessing

The INSOMNIA experiment uses original CICIDS-2017 weekday files only. Monday and Tuesday initialize the detector and label estimator. Wednesday, Thursday, and Friday form the temporal incremental stream. Labels are binary: `BENIGN = 0`, every attack label = `1`. Preprocessing statistics are fit only on Monday and Tuesday.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestCentroid

NON_LEARNING_METADATA_COLUMNS = {
    "flow id",
    "source ip",
    "src ip",
    "destination ip",
    "dst ip",
    "source port",
    "src port",
    "destination port",
    "dst port",
    "timestamp",
}


def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()


def is_benign_label(label: str) -> bool:
    normalized = str(label).strip().lower().replace(" ", "").replace("-", "").replace("_", "")
    return normalized in {"benign", "normal", "background"}


def binary_label_values(labels: pd.Series) -> np.ndarray:
    return np.where(clean_label_series(labels).apply(is_benign_label).to_numpy(), 0, 1).astype(np.int64)


def weekday_from_source(source_file: str) -> str | None:
    name = Path(str(source_file)).name.lower()
    for day in CICIDS_ORIGINAL_WEEKDAYS:
        if day in name:
            return day
    return None


def build_insomnia_day_split(df: pd.DataFrame):
    if "source_file" not in df.columns:
        raise ValueError("INSOMNIA requires source_file metadata from CSV loading.")

    ordered_df = df.copy()
    ordered_df["_source_day"] = ordered_df["source_file"].map(weekday_from_source)
    ordered_df["_source_day_rank"] = ordered_df["_source_day"].map(lambda day: CICIDS_ORIGINAL_WEEKDAYS.index(day) if day in CICIDS_ORIGINAL_WEEKDAYS else 999)
    ordered_df["_source_row_order"] = np.arange(len(ordered_df))
    ordered_df = ordered_df.sort_values(["_source_day_rank", "_source_row_order"], kind="mergesort")

    source_days = ordered_df["_source_day"]
    train_mask = source_days.isin(["monday", "tuesday"])
    stream_mask = source_days.isin(["wednesday", "thursday", "friday"])

    train_df = ordered_df[train_mask].drop(columns=["_source_day", "_source_day_rank", "_source_row_order"]).copy()
    stream_df = ordered_df[stream_mask].drop(columns=["_source_day", "_source_day_rank", "_source_row_order"]).copy()

    if train_df.empty or stream_df.empty:
        raise ValueError(
            f"Invalid CICIDS-2017 INSOMNIA split: train={train_df.shape}, stream={stream_df.shape}. "
            "Expected Monday+Tuesday for initialization and Wednesday-Friday for the stream."
        )

    train_files = sorted(train_df["source_file"].astype(str).unique(), key=lambda p: CICIDS_ORIGINAL_WEEKDAYS.index(weekday_from_source(p)))
    stream_files = sorted(stream_df["source_file"].astype(str).unique(), key=lambda p: CICIDS_ORIGINAL_WEEKDAYS.index(weekday_from_source(p)))

    print("Initialization / training files:")
    for file_name in train_files:
        print(" -", file_name)
    print("Incremental / test stream files:")
    for file_name in stream_files:
        print(" -", file_name)
    return train_df, stream_df, train_files, stream_files


def prepare_insomnia_data(train_df: pd.DataFrame, stream_df: pd.DataFrame):
    label_col = detect_label_column(train_df)
    drop_cols = {label_col, "source_file"}
    for col in train_df.columns:
        if col.strip().lower() in NON_LEARNING_METADATA_COLUMNS:
            drop_cols.add(col)

    feature_cols = [col for col in train_df.columns if col not in drop_cols]
    if not feature_cols:
        raise ValueError("No learning feature columns remain after dropping labels and metadata columns.")

    X_train_df = train_df[feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    X_stream_df = stream_df[feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_train_imputed = imputer.fit_transform(X_train_df)
    X_train_scaled = scaler.fit_transform(X_train_imputed).astype(np.float32)
    X_stream_scaled = scaler.transform(imputer.transform(X_stream_df)).astype(np.float32)

    y_train_values = binary_label_values(train_df[label_col])
    y_stream_values = binary_label_values(stream_df[label_col])
    stream_source_files = stream_df["source_file"].astype(str).to_numpy()

    print("Train shape:", X_train_scaled.shape, y_train_values.shape)
    print("Stream shape:", X_stream_scaled.shape, y_stream_values.shape)
    print("Dropped metadata columns:", sorted(col for col in drop_cols if col not in {label_col, "source_file"}))
    print("Learning feature count:", len(feature_cols))
    print("Training label distribution: BENIGN=0 / ATTACK=1")
    print(pd.Series(y_train_values).value_counts().sort_index())
    print("Stream label distribution: BENIGN=0 / ATTACK=1")
    print(pd.Series(y_stream_values).value_counts().sort_index())
    print("Imputer and scaler fitted on Monday+Tuesday only.")

    return {
        "X_train": X_train_scaled,
        "y_train": y_train_values,
        "X_stream": X_stream_scaled,
        "y_stream": y_stream_values,
        "stream_source_files": stream_source_files,
        "feature_cols": feature_cols,
        "label_col": label_col,
        "imputer": imputer,
        "scaler": scaler,
    }


train_df, stream_df, train_files, stream_files = build_insomnia_day_split(raw_df)
prepared = prepare_insomnia_data(train_df, stream_df)
X_train = prepared["X_train"]
y_train = prepared["y_train"]
X_stream = prepared["X_stream"]
y_stream = prepared["y_stream"]
stream_source_files = prepared["stream_source_files"]
NUM_FEATURES = X_train.shape[1]


## INSOMNIA Metrics and Windowing

True stream labels are used only for reporting metrics. They are never used for INSOMNIA-NC adaptation.

In [ ]:
WINDOW_SIZE = 50_000
SIGMA = 0.50
FAST_MODE = False
FAST_MODE_TRAINING_SEEDS = TRAINING_SEEDS[:1]
FAST_MODE_WEIGHT_INIT_NAMES = ["W1"]

ACTIVE_TRAINING_SEEDS = FAST_MODE_TRAINING_SEEDS if FAST_MODE else TRAINING_SEEDS
ACTIVE_WEIGHT_INIT_NAMES = FAST_MODE_WEIGHT_INIT_NAMES if FAST_MODE else ["W1", "W2", "W3"]
ACTIVE_WEIGHT_INIT_SEEDS = {name: WEIGHT_INIT_SEEDS[name] for name in ACTIVE_WEIGHT_INIT_NAMES}
RUN_CONFIGS = [
    (training_seed, weight_init_name, ACTIVE_WEIGHT_INIT_SEEDS[weight_init_name])
    for training_seed in ACTIVE_TRAINING_SEEDS
    for weight_init_name in ACTIVE_WEIGHT_INIT_NAMES
]
EXPECTED_FULL_RUNS = len(TRAINING_SEEDS) * 3

print("Window size:", WINDOW_SIZE)
print("Sigma:", SIGMA)
print("FAST_MODE:", FAST_MODE)
print("Seeds used:", ACTIVE_TRAINING_SEEDS)
print("Inits used:", ACTIVE_WEIGHT_INIT_NAMES)
print("Total runs:", len(RUN_CONFIGS))
if FAST_MODE:
    assert len(RUN_CONFIGS) == 1, "FAST_MODE must run exactly 1 seed x 1 init."
else:
    assert ACTIVE_TRAINING_SEEDS == [57, 305, 5, 9667, 405], "Full mode must use the mandated five seeds."
    assert ACTIVE_WEIGHT_INIT_NAMES == ["W1", "W2", "W3"], "Full mode must use W1, W2, and W3 only."
    assert len(RUN_CONFIGS) == EXPECTED_FULL_RUNS == 15, "Full mode must run exactly 15 seed/init configurations."


def make_stream_windows(X_values: np.ndarray, y_values: np.ndarray, source_files: np.ndarray, window_size: int):
    windows = []
    for start in range(0, len(X_values), window_size):
        end = min(start + window_size, len(X_values))
        windows.append({
            "window_index": len(windows),
            "start_index": start,
            "end_index": end,
            "X": X_values[start:end],
            "y": y_values[start:end],
            "source_files": source_files[start:end],
        })
    return windows


stream_windows = make_stream_windows(X_stream, y_stream, stream_source_files, WINDOW_SIZE)
print("Number of windows:", len(stream_windows))
for window in stream_windows[:5]:
    print(f"Window {window['window_index']}: rows {window['start_index']}:{window['end_index']}")
if len(stream_windows) > 5:
    print("...")


def evaluate_binary_window(y_true, probabilities, threshold: float = 0.5) -> dict:
    y_pred = (probabilities >= threshold).astype(np.int64)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def append_rows(rows: list[dict], csv_path: Path) -> None:
    if not rows:
        return
    pd.DataFrame(rows).to_csv(csv_path, mode="a", index=False, header=not csv_path.exists())


## INSOMNIA DNN Detector and NearestCentroid Label Estimator

The base detector is a three-hidden-layer fully connected DNN with ReLU activations, dropout, Xavier initialization, Adam, and binary logits. NearestCentroid is trained on the same Monday+Tuesday labels and then retrained on the enlarged labeled plus pseudo-labeled adaptation set after each stream window.

In [ ]:
class DNNDetector(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x)


def make_torch_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def apply_xavier_initialization(model: nn.Module, seed_tuple) -> None:
    layers_to_init = [m for m in model.modules() if isinstance(m, nn.Linear)]
    for index, layer in enumerate(layers_to_init):
        torch.manual_seed(seed_tuple[index % len(seed_tuple)])
        nn.init.xavier_uniform_(layer.weight)
        if layer.bias is not None:
            nn.init.zeros_(layer.bias)


def make_binary_loader(X_values: np.ndarray, y_values: np.ndarray, batch_size: int, seed: int, shuffle: bool = True):
    X_tensor = torch.tensor(X_values, dtype=torch.float32)
    y_tensor = torch.tensor(y_values, dtype=torch.float32).view(-1, 1)
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=make_torch_generator(seed) if shuffle else None)


def train_dnn_detector(
    model: nn.Module,
    X_values: np.ndarray,
    y_values: np.ndarray,
    training_seed: int,
    epochs: int,
    batch_size: int,
    lr: float,
):
    set_global_seed(training_seed)
    model.train()
    loader = make_binary_loader(X_values, y_values, batch_size=batch_size, seed=training_seed, shuffle=True)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
    return model


def predict_dnn_probabilities(model: nn.Module, X_values: np.ndarray, batch_size: int = 8192) -> np.ndarray:
    model.eval()
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(X_values), batch_size):
            batch = torch.tensor(X_values[start:start + batch_size], dtype=torch.float32, device=DEVICE)
            logits = model(batch)
            probabilities.append(torch.sigmoid(logits).detach().cpu().numpy().reshape(-1))
    return np.concatenate(probabilities) if probabilities else np.array([], dtype=np.float32)


INITIAL_MODEL_DIR = RESULTS_DIR / "initial_models"
INITIAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def build_initial_models(training_seed: int, weight_init_name: str, weight_init_tuple):
    print("=== INITIALIZATION PHASE ===")
    print(f"Training seed={training_seed}, weight_init={weight_init_name}, tuple={weight_init_tuple}")
    set_global_seed(training_seed)
    dnn = DNNDetector(NUM_FEATURES).to(DEVICE)
    apply_xavier_initialization(dnn, weight_init_tuple)
    dnn = train_dnn_detector(
        dnn,
        X_train,
        y_train,
        training_seed=training_seed,
        epochs=5,
        batch_size=512,
        lr=1e-3,
    )
    nc = NearestCentroid()
    nc.fit(X_train, y_train)

    dnn_path = INITIAL_MODEL_DIR / f"dnn_seed_{training_seed}_init_{weight_init_name}.pt"
    nc_path = INITIAL_MODEL_DIR / f"nearest_centroid_seed_{training_seed}_init_{weight_init_name}.pkl"
    torch.save(dnn.state_dict(), dnn_path)
    with open(nc_path, "wb") as handle:
        pickle.dump(nc, handle)
    print("Saved initial DNN detector to:", dnn_path)
    print("Saved initial NearestCentroid estimator to:", nc_path)
    return dnn, nc


def cleanup_after_run() -> None:
    gc.collect()
    torch.cuda.empty_cache()


## Window-Based INSOMNIA Experiments

`No-Update DNN` evaluates the initialized detector on every stream window without adaptation. `INSOMNIA-NC` selects uncertain samples, pseudo-labels them with NearestCentroid, fine-tunes the DNN on pseudo-labels, retrains NearestCentroid on the enlarged labeled plus pseudo-labeled set, and moves to the next window. True stream labels are used only inside metric computation.

In [ ]:
NO_UPDATE_RESULTS_PATH = RESULTS_DIR / "no_update_per_window.csv"
INSOMNIA_RESULTS_PATH = RESULTS_DIR / "insomnia_nc_per_window.csv"
FINAL_AGGREGATED_RESULTS_PATH = RESULTS_DIR / "aggregated_results.csv"

for output_path in [NO_UPDATE_RESULTS_PATH, INSOMNIA_RESULTS_PATH, FINAL_AGGREGATED_RESULTS_PATH]:
    if output_path.exists():
        output_path.unlink()


def no_update_baseline_run(training_seed: int, weight_init_name: str, weight_init_tuple) -> list[dict]:
    dnn, _ = build_initial_models(training_seed, weight_init_name, weight_init_tuple)
    rows = []
    for window in stream_windows:
        probabilities = predict_dnn_probabilities(dnn, window["X"])
        metrics = evaluate_binary_window(window["y"], probabilities)
        row = {
            "dataset_name": DATASET_NAME,
            "framework": FRAMEWORK,
            "method": NO_UPDATE_MODEL_NAME,
            "model_name": DNN_DETECTOR_NAME,
            "training_seed": training_seed,
            "weight_init": weight_init_name,
            "sigma": np.nan,
            "window_index": window["window_index"],
            "start_index": window["start_index"],
            "end_index": window["end_index"],
            "window_size": len(window["y"]),
            "selected_count": 0,
            "pseudo_label_benign_count": 0,
            "pseudo_label_attack_count": 0,
            **metrics,
        }
        rows.append(row)
        append_rows([row], NO_UPDATE_RESULTS_PATH)
    cleanup_after_run()
    return rows


def select_uncertain_indices(probabilities: np.ndarray, sigma: float) -> np.ndarray:
    if len(probabilities) == 0:
        return np.array([], dtype=np.int64)
    select_count = max(1, int(math.ceil(len(probabilities) * sigma)))
    uncertainty = np.abs(probabilities - 0.5)
    return np.argsort(uncertainty)[:select_count]


def insomnia_nc_run(training_seed: int, weight_init_name: str, weight_init_tuple, sigma: float) -> list[dict]:
    dnn, nc = build_initial_models(training_seed, weight_init_name, weight_init_tuple)
    pseudo_X_parts = []
    pseudo_y_parts = []
    rows = []

    for window in stream_windows:
        probabilities = predict_dnn_probabilities(dnn, window["X"])
        metrics = evaluate_binary_window(window["y"], probabilities)

        selected_indices = select_uncertain_indices(probabilities, sigma=sigma)
        selected_X = window["X"][selected_indices]
        # Correctness constraint: these pseudo-labels come from NearestCentroid, not from true stream labels.
        pseudo_labels = nc.predict(selected_X).astype(np.int64) if len(selected_X) else np.array([], dtype=np.int64)

        row = {
            "dataset_name": DATASET_NAME,
            "framework": FRAMEWORK,
            "method": INSOMNIA_MODEL_NAME,
            "model_name": DNN_DETECTOR_NAME,
            "training_seed": training_seed,
            "weight_init": weight_init_name,
            "sigma": sigma,
            "window_index": window["window_index"],
            "start_index": window["start_index"],
            "end_index": window["end_index"],
            "window_size": len(window["y"]),
            "selected_count": int(len(selected_indices)),
            "pseudo_label_benign_count": int((pseudo_labels == 0).sum()),
            "pseudo_label_attack_count": int((pseudo_labels == 1).sum()),
            **metrics,
        }
        rows.append(row)
        append_rows([row], INSOMNIA_RESULTS_PATH)

        if len(selected_X):
            pseudo_X_parts.append(selected_X)
            pseudo_y_parts.append(pseudo_labels)
            accumulated_pseudo_X = np.vstack(pseudo_X_parts)
            accumulated_pseudo_y = np.concatenate(pseudo_y_parts)

            dnn = train_dnn_detector(
                dnn,
                accumulated_pseudo_X,
                accumulated_pseudo_y,
                training_seed=training_seed + window["window_index"] + 1,
                epochs=1,
                batch_size=512,
                lr=2e-4,
            )
            nc.fit(
                np.vstack([X_train, accumulated_pseudo_X]),
                np.concatenate([y_train, accumulated_pseudo_y]),
            )

        cleanup_after_run()

    return rows


print("Seeds used:", ACTIVE_TRAINING_SEEDS)
print("Inits used:", ACTIVE_WEIGHT_INIT_NAMES)
print("Total runs:", len(RUN_CONFIGS))
print(f"total runs = {len(RUN_CONFIGS)}")
print("Number of windows:", len(stream_windows))

no_update_rows = []
insomnia_rows = []
for run_index, (training_seed, weight_init_name, weight_init_tuple) in enumerate(RUN_CONFIGS, start=1):
    print(f"Run {run_index}/{len(RUN_CONFIGS)}: seed={training_seed}, init={weight_init_name}")
    print(f"No-Update DNN: seed={training_seed}, init={weight_init_name}")
    no_update_rows.extend(no_update_baseline_run(training_seed, weight_init_name, weight_init_tuple))
    print(f"INSOMNIA-NC: sigma={SIGMA}, seed={training_seed}, init={weight_init_name}")
    insomnia_rows.extend(insomnia_nc_run(training_seed, weight_init_name, weight_init_tuple, sigma=SIGMA))

no_update_per_window_results = pd.DataFrame(no_update_rows)
insomnia_nc_per_window_results = pd.DataFrame(insomnia_rows)

print("=== NO-UPDATE BASELINE RESULTS ===")
display(no_update_per_window_results)
print("=== INSOMNIA-NC RESULTS ===")
display(insomnia_nc_per_window_results)


## INSOMNIA Reporting

Per-window results are reported first, followed by per-run summaries and aggregate mean/min/max/std statistics. F1 is the main metric.

In [ ]:
print("=== PER-WINDOW RESULTS ===")
per_window_results = pd.concat(
    [no_update_per_window_results, insomnia_nc_per_window_results],
    ignore_index=True,
)
display(per_window_results)

metric_cols = ["accuracy", "precision", "recall", "f1"]
per_run_group_cols = ["dataset_name", "framework", "method", "model_name", "training_seed", "weight_init", "sigma"]
per_run_results = per_window_results.groupby(per_run_group_cols, dropna=False)[metric_cols].mean().reset_index()
print("Per-run results:")
display(per_run_results)

aggregation_group_cols = ["dataset_name", "framework", "method", "model_name", "sigma"]
aggregated_results = per_run_results.groupby(aggregation_group_cols, dropna=False)[metric_cols].agg(
    ["mean", "min", "max", "std"]
).round(6)
aggregated_results.columns = [f"{metric}_{stat}" for metric, stat in aggregated_results.columns]
aggregated_results = aggregated_results.reset_index()

aggregated_results.to_csv(FINAL_AGGREGATED_RESULTS_PATH, index=False)

print("=== AGGREGATED RESULTS ===")
display(aggregated_results)
print("Saved no-update per-window results to:", NO_UPDATE_RESULTS_PATH)
print("Saved INSOMNIA-NC per-window results to:", INSOMNIA_RESULTS_PATH)
print("Saved final aggregated results to:", FINAL_AGGREGATED_RESULTS_PATH)
print("Seeds used:", ACTIVE_TRAINING_SEEDS)
print("Inits used:", ACTIVE_WEIGHT_INIT_NAMES)
print("Total runs:", len(RUN_CONFIGS))
print("Number of windows:", len(stream_windows))
print("=== DONE ===")

print("CSV result files saved under:", RESULTS_DIR)
for csv_path in sorted(RESULTS_DIR.glob("*.csv")):
    print(" -", csv_path)
